In [1]:
import requests
import psycopg
from dotenv import load_dotenv
import os

In [2]:
def getURL(url=''):
    if not url:
        print("URL not given!")
        return None

    try:
        response = requests.get(url)
        response.raise_for_status()
        tickets = response.json()
        return tickets

    except requests.exceptions.RequestException as e:
        print("HTTP request error", e)
        return None

In [3]:
#getURL('http://www.google.com/404')
#getURL()
tickets = getURL('https://jsonplaceholder.typicode.com/posts')

In [6]:
load_dotenv()


def run_sql_file(cursor, filename):
    try:
        with open(filename, "r") as f:
            sql_script = f.read()

        # multiple SQL queries
        statements = sql_script.split(";")

        for statement in statements:
            statement = statement.strip()
            if not statement:
                continue

            cursor.execute(statement)

            # Print the result only for SELECT queries
            if cursor.description:
                columns = [desc[0] for desc in cursor.description]
                print(" | ".join(columns))

                rows = cursor.fetchall()
                for row in rows:
                    print(row)

                print("-" * 40)

    except FileNotFoundError:
        print(f"File {filename} not found.")

    except Exception as e:
        print("Error executing SQL file:", e)


def save_tickets_to_db(data):
    if not data:
        print("No data available")
        return

    try:
        with psycopg.connect(
            host=os.getenv("DB_HOST"),
            dbname=os.getenv("DB_DATABASE"),
            user=os.getenv("DB_USER"),
            password=os.getenv("DB_PASSWORD"),
            port=os.getenv("DB_PORT")
        ) as connection:
            with connection.cursor() as cursor:

                # najprej ustvari tabelo
                run_sql_file(cursor, "create_table.sql")

                # potem vstavi podatke
                for ticket in data:
                    cursor.execute(
                        """
                        INSERT INTO tickets (ticket_id, user_id, title, body)
                        VALUES (%s, %s, %s, %s)
                        ON CONFLICT (ticket_id) DO NOTHING
                        """,
                        (
                            ticket.get("id"),
                            ticket.get("userId"),
                            ticket.get("title"),
                            ticket.get("body")
                        )
                    )

                print(f"{len(data)} records stored successfully.")
                print("-" * 40)
                print("-" * 40)

                # na koncu zaženi analizo in izpiši rezultate
                run_sql_file(cursor, "tickets_analyzer.sql")

    except psycopg.OperationalError as e:
        print("Connection error:", e)
    except psycopg.Error as e:
        print("Database error:", e)
    except Exception as e:
        print("Unexpected error:", e)

#naredi še execute_batch in COPY (hitrejši metodi)

In [7]:
save_tickets_to_db(tickets)

100 records stored successfully.
----------------------------------------
----------------------------------------
count
(100,)
----------------------------------------
count
(10,)
----------------------------------------
body
('aut dicta possimus sint mollitia voluptas commodi quo doloremque\niste corrupti reiciendis voluptatem eius rerum\nsit cumque quod eligendi laborum minima\nperferendis recusandae assumenda consectetur porro architecto ipsum ipsam',)
----------------------------------------
count
(10,)
----------------------------------------
user_id | number_of_tickets
(8, 10)
(10, 10)
(9, 10)
(7, 10)
(1, 10)
(5, 10)
(4, 10)
(2, 10)
(6, 10)
(3, 10)
----------------------------------------
